In [1]:
# ==========================================
# Notebook 07
# User Profile Vector Engine
# ==========================================

import pandas as pd
import numpy as np

In [2]:
items_df = pd.read_csv("../data/items.csv")

profiles_df = pd.read_csv("../data/item_profiles.csv")

interactions_df = pd.read_csv("../data/user_interactions.csv")

item_embeddings = np.load("../data/item_embeddings.npy")

In [3]:
print("Items:", len(items_df))

print()

print("Users:", interactions_df["user_id"].nunique())

print()

print("Embeddings Shape:", item_embeddings.shape)

Items: 10

Users: 5

Embeddings Shape: (10, 384)


In [4]:
interactions_df.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [5]:
interactions_df[interactions_df["user_id"] == 101]

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4


In [6]:
interactions_df[interactions_df["user_id"] == 101]

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4


In [7]:
item_index_mapping = dict(zip(profiles_df["item_id"], profiles_df.index))

item_index_mapping

{1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9}

In [8]:
user_profiles = []

for user_id in interactions_df["user_id"].unique():

    user_history = interactions_df[interactions_df["user_id"] == user_id]

    consumed_items = user_history["item_id"].tolist()

    consumed_embeddings = []

    for item_id in consumed_items:

        embedding_index = item_index_mapping[item_id]

        consumed_embeddings.append(item_embeddings[embedding_index])

    user_vector = np.mean(consumed_embeddings, axis=0)

    user_profiles.append({"user_id": user_id, "user_vector": user_vector})

In [9]:
len(user_profiles)

5

In [11]:
user_profiles

[{'user_id': 101,
  'user_vector': array([ 8.21680669e-03, -5.54117747e-03, -3.33765969e-02, -5.51803894e-02,
          1.49221160e-03, -3.07114329e-02,  3.97837199e-02,  2.70672385e-02,
         -3.90812904e-02,  5.19380579e-03, -2.84225568e-02,  1.56204440e-02,
          6.78217551e-03,  1.65611450e-02, -4.06162255e-02, -1.55382594e-02,
          1.55002857e-02, -5.71679473e-02, -1.22728469e-02,  2.30693202e-02,
         -2.65736431e-02, -4.54670079e-02, -1.17608309e-02,  4.17895876e-02,
          6.14151098e-02,  8.00634325e-02, -1.68158617e-02, -8.97147413e-03,
         -6.01649396e-02, -4.81769443e-02, -4.27343510e-03,  5.13095073e-02,
          2.62768548e-02,  3.71304154e-02,  3.33361663e-02,  2.95535326e-02,
          1.57578271e-02, -3.32237072e-02, -1.07257562e-02, -2.09950581e-02,
          6.45974502e-02, -4.30020094e-02, -5.15961349e-02,  2.45764907e-02,
          1.81658026e-02, -3.39043699e-02, -2.33578514e-02, -3.63786854e-02,
         -5.33905625e-03,  1.91127304e-02, 

In [12]:
user_profile_df = pd.DataFrame(
    {
        "user_id": [profile["user_id"] for profile in user_profiles],
        "user_vector": [profile["user_vector"] for profile in user_profiles],
    }
)

In [13]:
user_profile_df

,user_id,user_vector
0,101,"[0.008216807, -0.0055411775, -0.033376597, -0...."
1,102,"[0.016862007, -0.027616188, -0.023633063, -0.0..."
2,103,"[0.03836494, 0.009716912, -0.037663385, 0.0567..."
3,104,"[-0.06547663, -0.018401783, -0.004169222, -0.0..."
4,105,"[-0.02725611, 0.019708049, -0.055300195, -0.01..."


In [14]:
user_embedding_matrix = np.vstack(user_profile_df["user_vector"])

user_embedding_matrix.shape

(5, 384)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
user_similarity_matrix = cosine_similarity(user_embedding_matrix)

user_similarity_matrix

array([[1.0000001 , 0.89157   , 0.34353018, 0.3552066 , 0.6530695 ],
       [0.89157   , 0.99999976, 0.37069267, 0.31160516, 0.4499327 ],
       [0.34353018, 0.37069267, 0.99999994, 0.32846677, 0.3801741 ],
       [0.3552066 , 0.31160516, 0.32846677, 1.0000001 , 0.6677512 ],
       [0.6530695 , 0.4499327 , 0.3801741 , 0.6677512 , 1.0000002 ]],
      dtype=float32)

In [17]:
user_similarity_df = pd.DataFrame(
    user_similarity_matrix,
    index=user_profile_df["user_id"],
    columns=user_profile_df["user_id"],
)

user_similarity_df

user_id,101,102,103,104,105
user_id,,,,,
101,1.000000,0.891570,0.343530,0.355207,0.653069
102,0.891570,1.000000,0.370693,0.311605,0.449933
103,0.343530,0.370693,1.000000,0.328467,0.380174
104,0.355207,0.311605,0.328467,1.000000,0.667751
105,0.653069,0.449933,0.380174,0.667751,1.000000


In [18]:
target_user = 101

user_idx = user_profile_df[user_profile_df["user_id"] == target_user].index[0]

scores = user_similarity_matrix[user_idx]

scores

array([1.0000001 , 0.89157   , 0.34353018, 0.3552066 , 0.6530695 ],
      dtype=float32)

In [19]:
ranked_idx = np.argsort(scores)[::-1]

ranked_idx

array([0, 1, 4, 3, 2], dtype=int64)

In [20]:
for idx in ranked_idx:

    print()

    print(user_profile_df.iloc[idx]["user_id"])

    print(round(scores[idx], 4))


101
1.0

102
0.8916

105
0.6531

104
0.3552

103
0.3435


In [21]:
alpha = 0.8

In [22]:
current_profile = user_profiles[0]["user_vector"]

In [23]:
latest_item_embedding = item_embeddings[9]

In [24]:
updated_profile = alpha * current_profile + (1 - alpha) * latest_item_embedding

In [25]:
updated_profile.shape

(384,)

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
profile_shift = cosine_similarity(
    current_profile.reshape(1, -1), updated_profile.reshape(1, -1)
)[0][0]

profile_shift

0.9596695

In [28]:
user_embedding_df = pd.DataFrame(user_embedding_matrix)

In [29]:
user_embedding_df.insert(0, "user_id", user_profile_df["user_id"])

In [30]:
user_embedding_df.head()

,user_id,0,1,2,3,4,5,6,7,8,...,374,375,376,377,378,379,380,381,382,383
0,101,0.008217,-0.005541,-0.033377,-0.055180,0.001492,-0.030711,0.039784,0.027067,-0.039081,...,0.021838,-0.002430,-0.004756,0.026005,0.038743,0.031131,0.084350,-0.017390,0.037995,0.023191
1,102,0.016862,-0.027616,-0.023633,-0.049662,0.012153,-0.018081,0.039506,0.024467,-0.047100,...,0.027740,0.021260,-0.003182,0.060788,0.016970,-0.013508,0.104433,-0.034164,0.043555,0.001010
2,103,0.038365,0.009717,-0.037663,0.056794,-0.024490,-0.035980,0.017417,-0.000972,-0.096994,...,-0.037571,0.039922,0.047643,0.004742,-0.030438,-0.039578,0.142901,0.026361,-0.001394,-0.013149
3,104,-0.065477,-0.018402,-0.004169,-0.012562,0.032085,-0.035960,-0.010831,0.031132,-0.045164,...,0.049698,0.037003,0.035458,-0.063650,-0.009495,0.040789,0.053854,0.040240,-0.029143,0.004683
4,105,-0.027256,0.019708,-0.055300,-0.013605,0.013021,-0.079306,0.001470,0.003982,-0.042966,...,0.047351,0.006740,0.006698,-0.040654,0.053702,0.031907,0.073550,0.028771,0.013899,0.029564


In [31]:
np.save("../data/user_embeddings.npy", user_embedding_matrix)

In [32]:
user_embedding_df.to_csv("../data/user_profiles.csv", index=False)

In [33]:
loaded_profiles = pd.read_csv("../data/user_profiles.csv")

loaded_profiles.head()

,user_id,0,1,2,3,4,5,6,7,8,...,374,375,376,377,378,379,380,381,382,383
0,101,0.008217,-0.005541,-0.033377,-0.055180,0.001492,-0.030711,0.039784,0.027067,-0.039081,...,0.021838,-0.002430,-0.004756,0.026005,0.038743,0.031131,0.084350,-0.017390,0.037995,0.023191
1,102,0.016862,-0.027616,-0.023633,-0.049662,0.012153,-0.018081,0.039506,0.024467,-0.047100,...,0.027740,0.021260,-0.003182,0.060788,0.016970,-0.013508,0.104433,-0.034164,0.043555,0.001010
2,103,0.038365,0.009717,-0.037663,0.056794,-0.024490,-0.035980,0.017417,-0.000972,-0.096994,...,-0.037571,0.039922,0.047643,0.004742,-0.030438,-0.039578,0.142901,0.026361,-0.001394,-0.013149
3,104,-0.065477,-0.018402,-0.004169,-0.012562,0.032085,-0.035960,-0.010831,0.031132,-0.045164,...,0.049698,0.037003,0.035458,-0.063650,-0.009495,0.040789,0.053854,0.040240,-0.029143,0.004683
4,105,-0.027256,0.019708,-0.055300,-0.013605,0.013021,-0.079306,0.001470,0.003982,-0.042966,...,0.047351,0.006740,0.006698,-0.040654,0.053702,0.031907,0.073550,0.028771,0.013899,0.029564
